# Overlaying clustering results with Squidpy

Run using `squidpy_pypi` conda environment

In [ ]:
import squidpy as sq 
import matplotlib.pyplot as plt
import numpy as np 
import pandas as pd 
import os 
import tifffile 
import anndata as ad
from anndata import AnnData
import re
from typing import List, Tuple

# Functions

In [ ]:
def marker_overlay(
    adata: AnnData, 
    channel_names : List[str], 
    leiden_res: str, 
    markers_df: pd.DataFrame,
    cluster_id: str,
    library_id: str, 
    library_key: str = "library_key",
    seg_cell_id: str = "CellID",
    figsize: Tuple[int, int] = (15,15)
    ) -> None:

    corresponding_idxs = [int(markers_df.loc[markers_df["marker_name"] == channel_name, "row_num"].values[0]) for channel_name in channel_names]

    for corresponding_idx in corresponding_idxs:
        sq.pl.spatial_segment(
            adata[adata.obs[leiden_res] == cluster_id, :].copy(), 
            color       = leiden_res,
            library_id  = library_id,
            library_key = library_key,
            seg_cell_id = seg_cell_id,
            figsize= figsize,
            alpha= 1,
            seg_outline= True, 
            outline= False, 
            outline_color = "black",
            palette = "grey", 
            seg_contourpx = 3,
            img_channel = corresponding_idx
        )
        plt.title(f"{leiden_res} cluster: {cluster_id}\n{markers_df.loc[markers_df['row_num'] == corresponding_idx, 'marker_name'].values[0]}")
        plt.show()
    

# Initialize squidpy object

In [ ]:
img_dir = "/stor/scratch/Ehrlich/MxIF/aging_thymus/human_images/human_blocks/block6_patches/P49_2M/"
masked_img = tifffile.imread(os.path.join(img_dir, "masked_img.ome.tif"))
masked_img = masked_img.transpose((1, 2, 0)) # Re-order to match squidpy convention

markers_df = pd.read_csv(os.path.join(img_dir, "mask_markers.csv"))
channel_names = [re.sub("_mask", "", marker_name) for marker_name in markers_df["marker_name"].tolist()]
markers_df["marker_name"] = channel_names

# Starting w/ a small ROI
roi_library_id = "P49_2M"
x_min, x_max = (1_000, 2_000)
y_min, y_max = (2_750, 3_750)
img = sq.im.ImageContainer(masked_img[y_min:y_max, x_min:x_max, :], layer="phenotypic_markers", library_id= roi_library_id)
img.data["channels"] = channel_names
seg_mask = tifffile.imread(os.path.join(img_dir, "no_touch_nonfat_cell_mask.tif"))
img.add_img(seg_mask[y_min:y_max, x_min:x_max], layer="no_touch_nonfat", library_id = roi_library_id)
img["no_touch_nonfat"].attrs["segmentation"] = True
img
# The warning is fine.

# Load anndata object with the cluster levels 
    # These anndata objects will already have leiden clustering and normalization performed. 
    # This notebook will be used to combine clusters into cell types. 
binary_ad = ad.read_h5ad(os.path.join(img_dir, "binary_mask_median_leiden_clustering_XY_4000.h5ad"))
binary_ad.obs = binary_ad.obs.rename(columns={"leiden_0.50" : "binary_leiden_0.5", 
                                              "leiden_1.00" : "binary_leiden_1"})
filt_ad = ad.read_h5ad(os.path.join(img_dir, "binary_mask_filt_mean_leiden_clustering_XY_4000.h5ad"))
filt_ad.obs = filt_ad.obs.rename(columns={"leiden_1.00" : "filt_leiden_1", 
                                          "leiden_1.50" : "filt_leiden_1.5", 
                                          "leiden_1.75" : "filt_leiden_1.75"})
shared_cellids = list(set(filt_ad.obs["CellID"].tolist()) & set(binary_ad.obs["CellID"].tolist()))
kept_cells = filt_ad.obs["CellID"].isin(shared_cellids)
merge_ad = filt_ad.copy()
merge_ad.obs = pd.merge(filt_ad.obs, binary_ad.obs, on= ["CellID", "Area", "X_centroid", "Y_centroid"], how= "left")
merge_ad = merge_ad[merge_ad.obs["binary_leiden_0.5"].notna().values, :].copy()


# Adding imaging data to anndata 
merge_ad.obsm["spatial"] = merge_ad.obs.loc[:, ["X_centroid", "Y_centroid"]].values

# image containers store mxif data as Xarray DataArray, but the anndata.uns["spatial"] images have to be a numpy array for Squidpy functions to work 

merge_ad.uns["spatial"] = {
    roi_library_id : {"images": {"hires": img["phenotypic_markers"].to_numpy()[:, :, 0, :]},
                           "scalefactors": {"tissue_hires_scalef": 1.0,
                                            'spot_diameter_fullres' :  10.0}
    }
}

# Now that I have defined the structure of the .un["spatial"] data. I can add another image alongside "hires".
merge_ad.uns["spatial"][roi_library_id]["images"]["segmentation"] = img["no_touch_nonfat"].to_numpy()[:, :, 0, 0]
merge_ad.obs["library_key"] = "P49_2M"

# filt_leiden_1 cell typing 

Each subheading has the markers used to identify that cluster

In [ ]:
leiden_res = "filt_leiden_1"

## Cluster 0

K10+ TEC, but some of them are missing. 

In [ ]:
channel_names_to_show = ["K10"]
corresponding_idxs = [int(markers_df.loc[markers_df["marker_name"] == channel_name, "row_num"].values[0]) for channel_name in channel_names_to_show]
cluster_id= "0"

sq.pl.spatial_segment(
    merge_ad[merge_ad.obs[leiden_res] == cluster_id, :].copy(), 
    color       = leiden_res,
    library_id  = "P49_2M",
    library_key = "library_key",
    seg_cell_id = "CellID",
    figsize= (15, 15),
    alpha= 1,
    seg_outline= True, 
    outline= True, 
    seg_contourpx = 3,
    img_channel = corresponding_idxs 
)
plt.title(f"{leiden_res} cluster: {cluster_id}")
plt.show()

## Cluster 1

In [ ]:
channel_names_to_show = ["CD3e", "CD8", "CD4", "Ki67"]
corresponding_idxs = [int(markers_df.loc[markers_df["marker_name"] == channel_name, "row_num"].values[0]) for channel_name in channel_names_to_show]
cluster_id= "1"

for corresponding_idx in corresponding_idxs:
    sq.pl.spatial_segment(
        merge_ad[merge_ad.obs[leiden_res] == cluster_id, :].copy(), 
        color       = leiden_res,
        library_id  = "P49_2M",
        library_key = "library_key",
        seg_cell_id = "CellID",
        figsize= (15, 15),
        alpha= 1,
        seg_outline= True, 
        outline= True, 
        seg_contourpx = 3,
        img_channel = corresponding_idx
    )
    plt.title(f"{leiden_res} cluster: {cluster_id}\n{markers_df.loc[markers_df['row_num'] == corresponding_idx, 'marker_name'].values[0]}")
    plt.show()

## Cluster 2

In [ ]:
channel_names_to_show = ["CD11c", "FOXP3", "K10"]
corresponding_idxs = [int(markers_df.loc[markers_df["marker_name"] == channel_name, "row_num"].values[0]) for channel_name in channel_names_to_show]
cluster_id= "2"


sq.pl.spatial_segment(
    merge_ad[merge_ad.obs[leiden_res] == cluster_id, :].copy(), 
    color       = leiden_res,
    library_id  = "P49_2M",
    library_key = "library_key",
    seg_cell_id = "CellID",
    figsize= (15, 15),
    alpha= 1,
    seg_outline= True, 
    outline= True, 
    seg_contourpx = 3,
    img_channel = corresponding_idxs 
)
plt.title(f"{leiden_res} cluster: {cluster_id}")
plt.show()

## Cluster 3 

In [ ]:
channel_names_to_show = ["CD20", "Pan-Cytokeratin", "CD44"]
corresponding_idxs = [int(markers_df.loc[markers_df["marker_name"] == channel_name, "row_num"].values[0]) for channel_name in channel_names_to_show]
cluster_id= "3"


for corresponding_idx in corresponding_idxs:
    sq.pl.spatial_segment(
        merge_ad[merge_ad.obs[leiden_res] == cluster_id, :].copy(), 
        color       = leiden_res,
        library_id  = "P49_2M",
        library_key = "library_key",
        seg_cell_id = "CellID",
        figsize= (15, 15),
        alpha= 1,
        seg_outline= True, 
        outline= True, 
        seg_contourpx = 3,
        img_channel = corresponding_idx
    )
    plt.title(f"{leiden_res} cluster: {cluster_id}\n{markers_df.loc[markers_df['row_num'] == corresponding_idx, 'marker_name'].values[0]}")
    plt.show()

## Cluster 5

In [ ]:
channel_names_to_show = ["CD4", "CD8", "CD3e", "Ki67"]
corresponding_idxs = [int(markers_df.loc[markers_df["marker_name"] == channel_name, "row_num"].values[0]) for channel_name in channel_names_to_show]
cluster_id= "5"


for corresponding_idx in corresponding_idxs:
    sq.pl.spatial_segment(
        merge_ad[merge_ad.obs[leiden_res] == cluster_id, :].copy(), 
        color       = leiden_res,
        library_id  = "P49_2M",
        library_key = "library_key",
        seg_cell_id = "CellID",
        figsize= (15, 15),
        alpha= 1,
        seg_outline= True, 
        outline= True, 
        seg_contourpx = 3,
        img_channel = corresponding_idx
    )
    plt.title(f"{leiden_res} cluster: {cluster_id}\n{markers_df.loc[markers_df['row_num'] == corresponding_idx, 'marker_name'].values[0]}")
    plt.show()

## Cluster 6

In [ ]:
channel_names_to_show = ["CD68", "CD11c", "Pan-Cytokeratin"]
corresponding_idxs = [int(markers_df.loc[markers_df["marker_name"] == channel_name, "row_num"].values[0]) for channel_name in channel_names_to_show]
cluster_id= "6"

for corresponding_idx in corresponding_idxs:
    sq.pl.spatial_segment(
        merge_ad[merge_ad.obs[leiden_res] == cluster_id, :].copy(), 
        color       = leiden_res,
        library_id  = "P49_2M",
        library_key = "library_key",
        seg_cell_id = "CellID",
        figsize= (15, 15),
        alpha= 1,
        seg_outline= True, 
        outline= True, 
        seg_contourpx = 3,
        img_channel = corresponding_idx
    )
    plt.title(f"{leiden_res} cluster: {cluster_id}\n{markers_df.loc[markers_df['row_num'] == corresponding_idx, 'marker_name'].values[0]}")
    plt.show()

## Cluster 10

In [ ]:
channel_names_to_show = ["CD4", "CD8", "CD3e", "Ki67"]
corresponding_idxs = [int(markers_df.loc[markers_df["marker_name"] == channel_name, "row_num"].values[0]) for channel_name in channel_names_to_show]
cluster_id= "10"


for corresponding_idx in corresponding_idxs:
    sq.pl.spatial_segment(
        merge_ad[merge_ad.obs[leiden_res] == cluster_id, :].copy(), 
        color       = leiden_res,
        library_id  = "P49_2M",
        library_key = "library_key",
        seg_cell_id = "CellID",
        figsize= (15, 15),
        alpha= 1,
        seg_outline= True, 
        outline= True, 
        seg_contourpx = 3,
        img_channel = corresponding_idx
    )
    plt.title(f"{leiden_res} cluster: {cluster_id}\n{markers_df.loc[markers_df['row_num'] == corresponding_idx, 'marker_name'].values[0]}")
    plt.show()

## Cluster 11

In [ ]:
channel_names_to_show = ["CD8", "CD4"]
corresponding_idxs = [int(markers_df.loc[markers_df["marker_name"] == channel_name, "row_num"].values[0]) for channel_name in channel_names_to_show]
cluster_id= "11"

for corresponding_idx in corresponding_idxs:
    sq.pl.spatial_segment(
        merge_ad[merge_ad.obs[leiden_res] == cluster_id, :].copy(), 
        color       = leiden_res,
        library_id  = "P49_2M",
        library_key = "library_key",
        seg_cell_id = "CellID",
        figsize= (15, 15),
        alpha= 1,
        seg_outline= True, 
        outline= True, 
        seg_contourpx = 3,
        img_channel = corresponding_idx
    )
    plt.title(f"{leiden_res} cluster: {cluster_id}\n{markers_df.loc[markers_df['row_num'] == corresponding_idx, 'marker_name'].values[0]}")
    plt.show()

## Cluster 13

In [ ]:
channel_names_to_show = ["CD8", "CD4", "Ki67"]
corresponding_idxs = [int(markers_df.loc[markers_df["marker_name"] == channel_name, "row_num"].values[0]) for channel_name in channel_names_to_show]
cluster_id= "13"

for corresponding_idx in corresponding_idxs:
    sq.pl.spatial_segment(
        merge_ad[merge_ad.obs[leiden_res] == cluster_id, :].copy(), 
        color       = leiden_res,
        library_id  = "P49_2M",
        library_key = "library_key",
        seg_cell_id = "CellID",
        figsize= (15, 15),
        alpha= 1,
        seg_outline= True, 
        outline= True, 
        seg_contourpx = 3,
        img_channel = corresponding_idx
    )
    plt.title(f"{leiden_res} cluster: {cluster_id}\n{markers_df.loc[markers_df['row_num'] == corresponding_idx, 'marker_name'].values[0]}")
    plt.show()

## Cluster 16

In [ ]:
channel_names_to_show = ["CD8", "CD4", "CD34", "b-Catenin1"]
corresponding_idxs = [int(markers_df.loc[markers_df["marker_name"] == channel_name, "row_num"].values[0]) for channel_name in channel_names_to_show]
cluster_id= "16"

for corresponding_idx in corresponding_idxs:
    sq.pl.spatial_segment(
        merge_ad[merge_ad.obs[leiden_res] == cluster_id, :].copy(), 
        color       = leiden_res,
        library_id  = "P49_2M",
        library_key = "library_key",
        seg_cell_id = "CellID",
        figsize= (15, 15),
        alpha= 1,
        seg_outline= True, 
        outline= True, 
        seg_contourpx = 3,
        img_channel = corresponding_idx
    )
    plt.title(f"{leiden_res} cluster: {cluster_id}\n{markers_df.loc[markers_df['row_num'] == corresponding_idx, 'marker_name'].values[0]}")
    plt.show()

## Cluster 17

In [ ]:
# channel_names_to_show = ["FOXP3", "CD4", "CD3e"]
channel_names_to_show = ["FOXP3"]
corresponding_idxs = [int(markers_df.loc[markers_df["marker_name"] == channel_name, "row_num"].values[0]) for channel_name in channel_names_to_show]
cluster_id= "17"

sq.pl.spatial_segment(
    merge_ad[merge_ad.obs[leiden_res] == cluster_id, :].copy(), 
    color       = leiden_res,
    library_id  = "P49_2M",
    library_key = "library_key",
    seg_cell_id = "CellID",
    figsize= (15, 15),
    alpha= 1,
    seg_outline= True, 
    outline= True, 
    seg_contourpx = 3,
    img_channel = corresponding_idxs 
)
plt.title(f"{leiden_res} cluster: {cluster_id}")
plt.show()

## Cluster 18

In [ ]:
channel_names = ["K5", "K14", "K8"]
corresponding_idxs = [int(markers_df.loc[markers_df["marker_name"] == channel_name, "row_num"].values[0]) for channel_name in channel_names]
img.show(channelwise = False, channel= corresponding_idxs, layer= "phenotypic_markers", figsize = (15,15))

In [ ]:
marker_overlay(
    adata = merge_ad, 
    channel_names= ["CD3e", "CD4", "CD8", "Pan-Cytokeratin", "K8", "K5", "K14", "K10", "b-Catenin1", "Ki67"],
    leiden_res = leiden_res, 
    markers_df = markers_df,
    cluster_id = "23",
    library_id = "P49_2M"
) 

## Cluster 20

In [ ]:
channel_names_to_show = ["CD4", "CD8", "Ki67","CD3e", "Bcl-2"]
corresponding_idxs = [int(markers_df.loc[markers_df["marker_name"] == channel_name, "row_num"].values[0]) for channel_name in channel_names_to_show]
cluster_id= "20"

for corresponding_idx in corresponding_idxs:
    sq.pl.spatial_segment(
        merge_ad[merge_ad.obs[leiden_res] == cluster_id, :].copy(), 
        color       = leiden_res,
        library_id  = "P49_2M",
        library_key = "library_key",
        seg_cell_id = "CellID",
        figsize= (15, 15),
        alpha= 1,
        seg_outline= True, 
        outline= True, 
        seg_contourpx = 3,
        img_channel = corresponding_idx
    )
    plt.title(f"{leiden_res} cluster: {cluster_id}\n{markers_df.loc[markers_df['row_num'] == corresponding_idx, 'marker_name'].values[0]}")
    plt.show()

## Cluster 22

In [ ]:
channel_names_to_show = ["CD141", "CD34", "CD3e"]
corresponding_idxs = [int(markers_df.loc[markers_df["marker_name"] == channel_name, "row_num"].values[0]) for channel_name in channel_names_to_show]
cluster_id= "22"

for corresponding_idx in corresponding_idxs:
    sq.pl.spatial_segment(
        merge_ad[merge_ad.obs[leiden_res] == cluster_id, :].copy(), 
        color       = leiden_res,
        library_id  = "P49_2M",
        library_key = "library_key",
        seg_cell_id = "CellID",
        figsize= (15, 15),
        alpha= 1,
        seg_outline= True, 
        outline= True, 
        seg_contourpx = 3,
        img_channel = corresponding_idx
    )
    plt.title(f"{leiden_res} cluster: {cluster_id}\n{markers_df.loc[markers_df['row_num'] == corresponding_idx, 'marker_name'].values[0]}")
    plt.show()

## Cluster 23

In [ ]:
marker_overlay(
    adata = merge_ad, 
    channel_names= ["Collagen IV","CD3e", "Bcl-2", "K8", "Pan-Cytokeratin", "b-Catenin1", "Ki67"],
    leiden_res = leiden_res, 
    markers_df = markers_df,
    cluster_id = "23",
    library_id = "P49_2M"
) 

## Cluster 24

In [ ]:
channel_names_to_show = ["CD31", "Collagen IV"]
corresponding_idxs = [int(markers_df.loc[markers_df["marker_name"] == channel_name, "row_num"].values[0]) for channel_name in channel_names_to_show]
cluster_id= "24"

for corresponding_idx in corresponding_idxs:
    sq.pl.spatial_segment(
        merge_ad[merge_ad.obs[leiden_res] == cluster_id, :].copy(), 
        color       = leiden_res,
        library_id  = "P49_2M",
        library_key = "library_key",
        seg_cell_id = "CellID",
        figsize= (15, 15),
        alpha= 1,
        seg_outline= True, 
        outline= True, 
        seg_contourpx = 3,
        img_channel = corresponding_idx
    )
    plt.title(f"{leiden_res} cluster: {cluster_id}\n{markers_df.loc[markers_df['row_num'] == corresponding_idx, 'marker_name'].values[0]}")
    plt.show()

## Cluster 25

In [ ]:
channel_names_to_show = ["b-Catenin1", "Pan-Cytokeratin", "Ki67","CD3e"]
corresponding_idxs = [int(markers_df.loc[markers_df["marker_name"] == channel_name, "row_num"].values[0]) for channel_name in channel_names_to_show]
cluster_id= "25"

for corresponding_idx in corresponding_idxs:
    sq.pl.spatial_segment(
        merge_ad[merge_ad.obs[leiden_res] == cluster_id, :].copy(), 
        color       = leiden_res,
        library_id  = "P49_2M",
        library_key = "library_key",
        seg_cell_id = "CellID",
        figsize= (15, 15),
        alpha= 1,
        seg_outline= True, 
        outline= True, 
        seg_contourpx = 3,
        img_channel = corresponding_idx
    )
    plt.title(f"{leiden_res} cluster: {cluster_id}\n{markers_df.loc[markers_df['row_num'] == corresponding_idx, 'marker_name'].values[0]}")
    plt.show()

# Binary_leiden_0.25

In [ ]:
leiden_res = "leiden_0.25"

## Cluster 0

In [ ]:
# channel_names_to_show = ["FOXP3", "CD4", "CD3e"]
channel_names_to_show = ["FOXP3", "Collagen IV", "K8"]
# channel_names_to_show = ["FOXP3", ]
corresponding_idxs = [int(markers_df.loc[markers_df["marker_name"] == channel_name, "row_num"].values[0]) for channel_name in channel_names_to_show]
cluster_id= "0"

sq.pl.spatial_segment(
    merge_ad[merge_ad.obs[leiden_res] == cluster_id, :].copy(), 
    color       = leiden_res,
    library_id  = "P49_2M",
    library_key = "library_key",
    seg_cell_id = "CellID",
    figsize= (15, 15),
    alpha= 1,
    seg_outline= True, 
    outline= True, 
    seg_contourpx = 3,
    img_channel = corresponding_idxs 
)
plt.title(f"{leiden_res} cluster: {cluster_id}")
plt.show()

## Cluster 1

In [ ]:
# channel_names_to_show = ["CD68", "Vimentin", "CD11c"]
channel_names_to_show = ["CD68"]
corresponding_idxs = [int(markers_df.loc[markers_df["marker_name"] == channel_name, "row_num"].values[0]) for channel_name in channel_names_to_show]
cluster_id= "1"

sq.pl.spatial_segment(
    merge_ad[merge_ad.obs[leiden_res] == cluster_id, :].copy(), 
    color       = leiden_res,
    library_id  = "P49_2M",
    library_key = "library_key",
    seg_cell_id = "CellID",
    figsize= (15, 15),
    alpha= 1,
    seg_outline= True, 
    outline= True, 
    seg_contourpx = 3,
    img_channel = corresponding_idxs 
)
plt.title(f"{leiden_res} cluster: {cluster_id}")
plt.show()

## Cluster 2

This is a bad segmentation cluster. These are DPs for the most part. This is probably an artifact of the background removal. 

In [ ]:
# channel_names_to_show = ["CD68", "Vimentin", "CD11c"]
channel_names_to_show = ["CD4", "CD8", "CD3e"]
corresponding_idxs = [int(markers_df.loc[markers_df["marker_name"] == channel_name, "row_num"].values[0]) for channel_name in channel_names_to_show]
cluster_id= "2"

sq.pl.spatial_segment(
    merge_ad[merge_ad.obs[leiden_res] == cluster_id, :].copy(), 
    color       = leiden_res,
    library_id  = "P49_2M",
    library_key = "library_key",
    seg_cell_id = "CellID",
    figsize= (15, 15),
    alpha= 1,
    seg_outline= True, 
    outline= True, 
    seg_contourpx = 3,
    img_channel = corresponding_idxs 
)
plt.title(f"{leiden_res} cluster: {cluster_id}")
plt.show()

## Cluster 3 

In [ ]:
channel_names_to_show = ["CD8", "CD4"]
corresponding_idxs = [int(markers_df.loc[markers_df["marker_name"] == channel_name, "row_num"].values[0]) for channel_name in channel_names_to_show]
cluster_id= "3"

for corresponding_idx in corresponding_idxs:
    sq.pl.spatial_segment(
        merge_ad[merge_ad.obs[leiden_res] == cluster_id, :].copy(), 
        color       = leiden_res,
        library_id  = "P49_2M",
        library_key = "library_key",
        seg_cell_id = "CellID",
        figsize= (15, 15),
        alpha= 1,
        seg_outline= True, 
        outline= True, 
        seg_contourpx = 3,
        img_channel = corresponding_idx
    )
    plt.title(f"{leiden_res} cluster: {cluster_id}")
    plt.show()

## Cluster 4

In [ ]:
channel_names_to_show = ["CD8", "CD4", "Bcl-2"]
corresponding_idxs = [int(markers_df.loc[markers_df["marker_name"] == channel_name, "row_num"].values[0]) for channel_name in channel_names_to_show]
cluster_id= "4"

for corresponding_idx in corresponding_idxs:
    sq.pl.spatial_segment(
        merge_ad[merge_ad.obs[leiden_res] == cluster_id, :].copy(), 
        color       = leiden_res,
        library_id  = "P49_2M",
        library_key = "library_key",
        seg_cell_id = "CellID",
        figsize= (15, 15),
        alpha= 1,
        seg_outline= True, 
        outline= True, 
        seg_contourpx = 3,
        img_channel = corresponding_idx
    )
    plt.title(f"{leiden_res} cluster: {cluster_id}")
    plt.show()

## Cluster 5

In [ ]:
channel_names_to_show = ["CD8", "CD3e", "CD4", "Bcl-2"]
corresponding_idxs = [int(markers_df.loc[markers_df["marker_name"] == channel_name, "row_num"].values[0]) for channel_name in channel_names_to_show]
cluster_id= "5"

for corresponding_idx in corresponding_idxs:
    sq.pl.spatial_segment(
        merge_ad[merge_ad.obs[leiden_res] == cluster_id, :].copy(), 
        color       = leiden_res,
        library_id  = "P49_2M",
        library_key = "library_key",
        seg_cell_id = "CellID",
        figsize= (15, 15),
        alpha= 1,
        seg_outline= True, 
        outline= True, 
        seg_contourpx = 3,
        img_channel = corresponding_idx
    )
    plt.title(f"{leiden_res} cluster: {cluster_id}")
    plt.show()

## Cluster 6

In [ ]:
channel_names_to_show = ["CD8", "CD4", "CD3e", "Ki67"]
corresponding_idxs = [int(markers_df.loc[markers_df["marker_name"] == channel_name, "row_num"].values[0]) for channel_name in channel_names_to_show]
cluster_id= "6"

for corresponding_idx in corresponding_idxs:
    sq.pl.spatial_segment(
        merge_ad[merge_ad.obs[leiden_res] == cluster_id, :].copy(), 
        color       = leiden_res,
        library_id  = "P49_2M",
        library_key = "library_key",
        seg_cell_id = "CellID",
        figsize= (15, 15),
        alpha= 1,
        seg_outline= True, 
        outline= True, 
        seg_contourpx = 3,
        img_channel = corresponding_idx
    )
    plt.title(f"{leiden_res} cluster: {cluster_id}")
    plt.show()

## Cluster 7

In [ ]:
channel_names_to_show = ["CD8", "CD4", "CD3e", "Ki67"]
corresponding_idxs = [int(markers_df.loc[markers_df["marker_name"] == channel_name, "row_num"].values[0]) for channel_name in channel_names_to_show]
cluster_id= "7"

for corresponding_idx in corresponding_idxs:
    sq.pl.spatial_segment(
        merge_ad[merge_ad.obs[leiden_res] == cluster_id, :].copy(), 
        color       = leiden_res,
        library_id  = "P49_2M",
        library_key = "library_key",
        seg_cell_id = "CellID",
        figsize= (15, 15),
        alpha= 1,
        seg_outline= True, 
        outline= True, 
        seg_contourpx = 3,
        img_channel = corresponding_idx
    )
    plt.title(f"{leiden_res} cluster: {cluster_id}")
    plt.show()

## Cluster 8

In [ ]:
channel_names_to_show = ["CD8", "CD4", "CD3e", "Ki67"]
corresponding_idxs = [int(markers_df.loc[markers_df["marker_name"] == channel_name, "row_num"].values[0]) for channel_name in channel_names_to_show]
cluster_id= "8"

for corresponding_idx in corresponding_idxs:
    sq.pl.spatial_segment(
        merge_ad[merge_ad.obs[leiden_res] == cluster_id, :].copy(), 
        color       = leiden_res,
        library_id  = "P49_2M",
        library_key = "library_key",
        seg_cell_id = "CellID",
        figsize= (15, 15),
        alpha= 1,
        seg_outline= True, 
        outline= True, 
        seg_contourpx = 3,
        img_channel = corresponding_idx
    )
    plt.title(f"{leiden_res} cluster: {cluster_id}")
    plt.show()

## Cluster 9 

In [ ]:
channel_names_to_show = ["CD3e", "CD8", "CD4", "Bcl-2"]
corresponding_idxs = [int(markers_df.loc[markers_df["marker_name"] == channel_name, "row_num"].values[0]) for channel_name in channel_names_to_show]
cluster_id= "9"

for corresponding_idx in corresponding_idxs:
    sq.pl.spatial_segment(
        merge_ad[merge_ad.obs[leiden_res] == cluster_id, :].copy(), 
        color       = leiden_res,
        library_id  = "P49_2M",
        library_key = "library_key",
        seg_cell_id = "CellID",
        figsize= (15, 15),
        alpha= 1,
        seg_outline= True, 
        outline= True, 
        seg_contourpx = 3,
        img_channel = corresponding_idx
    )
    plt.title(f"{leiden_res} cluster: {cluster_id}")
    plt.show()

## Cluster 10

In [ ]:
channel_names_to_show = ["CD3e", "CD8", "CD4", "Bcl-2"]
corresponding_idxs = [int(markers_df.loc[markers_df["marker_name"] == channel_name, "row_num"].values[0]) for channel_name in channel_names_to_show]
cluster_id= "10"

for corresponding_idx in corresponding_idxs:
    sq.pl.spatial_segment(
        merge_ad[merge_ad.obs[leiden_res] == cluster_id, :].copy(), 
        color       = leiden_res,
        library_id  = "P49_2M",
        library_key = "library_key",
        seg_cell_id = "CellID",
        figsize= (15, 15),
        alpha= 1,
        seg_outline= True, 
        outline= True, 
        seg_contourpx = 3,
        img_channel = corresponding_idx
    )
    plt.title(f"{leiden_res} cluster: {cluster_id}")
    plt.show()

## Cluster 14

In [ ]:
channel_names_to_show = ["CD44", "CD8", "CD20", "K10", "CD11c", "CD68"]
corresponding_idxs = [int(markers_df.loc[markers_df["marker_name"] == channel_name, "row_num"].values[0]) for channel_name in channel_names_to_show]
cluster_id= "14"

for corresponding_idx in corresponding_idxs:
    sq.pl.spatial_segment(
        merge_ad[merge_ad.obs[leiden_res] == cluster_id, :].copy(), 
        color       = leiden_res,
        library_id  = "P49_2M",
        library_key = "library_key",
        seg_cell_id = "CellID",
        figsize= (15, 15),
        alpha= 1,
        seg_outline= True, 
        outline= True, 
        seg_contourpx = 3,
        img_channel = corresponding_idx
    )
    plt.title(f"{leiden_res} cluster: {cluster_id}")
    plt.show()

## Cluster 15

In [ ]:
channel_names_to_show = ["Vimentin", "CD8", "CD3e"]
corresponding_idxs = [int(markers_df.loc[markers_df["marker_name"] == channel_name, "row_num"].values[0]) for channel_name in channel_names_to_show]
cluster_id= "15"

for corresponding_idx in corresponding_idxs:
    sq.pl.spatial_segment(
        merge_ad[merge_ad.obs[leiden_res] == cluster_id, :].copy(), 
        color       = leiden_res,
        library_id  = "P49_2M",
        library_key = "library_key",
        seg_cell_id = "CellID",
        figsize= (15, 15),
        alpha= 1,
        seg_outline= True, 
        outline= True, 
        seg_contourpx = 3,
        img_channel = corresponding_idx
    )
    plt.title(f"{leiden_res} cluster: {cluster_id}")
    plt.show()

## Cluster 17

In [ ]:
channel_names_to_show = ["CD141", "CD34", "Vimentin", "CD31", 'CD11c', "Pan-Cytokeratin", "b-Catenin1"]
corresponding_idxs = [int(markers_df.loc[markers_df["marker_name"] == channel_name, "row_num"].values[0]) for channel_name in channel_names_to_show]
cluster_id= "17"

for corresponding_idx in corresponding_idxs:
    sq.pl.spatial_segment(
        merge_ad[merge_ad.obs[leiden_res] == cluster_id, :].copy(), 
        color       = leiden_res,
        library_id  = "P49_2M",
        library_key = "library_key",
        seg_cell_id = "CellID",
        figsize= (15, 15),
        alpha= 1,
        seg_outline= True, 
        outline= True, 
        seg_contourpx = 3,
        img_channel = corresponding_idx
    )
    plt.title(f"{leiden_res} cluster: {cluster_id}")
    plt.show()

## Cluster 22

In [ ]:
channel_names_to_show = ["K5", "K14"]
corresponding_idxs = [int(markers_df.loc[markers_df["marker_name"] == channel_name, "row_num"].values[0]) for channel_name in channel_names_to_show]
cluster_id= "22"

for corresponding_idx in corresponding_idxs:
    sq.pl.spatial_segment(
        merge_ad[merge_ad.obs[leiden_res] == cluster_id, :].copy(), 
        color       = leiden_res,
        library_id  = "P49_2M",
        library_key = "library_key",
        seg_cell_id = "CellID",
        figsize= (15, 15),
        alpha= 1,
        seg_outline= True, 
        outline= True, 
        seg_contourpx = 3,
        img_channel = corresponding_idx
    )
    plt.title(f"{leiden_res} cluster: {cluster_id}")
    plt.show()

## Cluster 30

In [ ]:
marker_overlay(
    adata = merge_ad, 
    channel_names= ["Pan-Cytokeratin", "CD4", "CD8"],
    leiden_res = leiden_res, 
    markers_df = markers_df,
    cluster_id = "30",
    library_id = "P49_2M"
) 

## Cluster 31

In [ ]:
marker_overlay(
    adata = merge_ad, 
    channel_names= ["Pan-Cytokeratin", "CD4", "CD8"],
    leiden_res = leiden_res, 
    markers_df = markers_df,
    cluster_id = "31",
    library_id = "P49_2M"
) 

# Mean no filtering but not touching mask clusters

In [ ]:
leiden_res = "filt_leiden_1"

In [ ]:
marker_overlay(
    adata = merge_ad, 
    channel_names= ["K5", "K14"],
    leiden_res = leiden_res, 
    markers_df = markers_df,
    cluster_id = "22",
    library_id = "P49_2M"
) 